# 01. Spatiotemporal tensor geometry

![Video tensors become tubelet tokens](../images/01_spatiotemporal_tensor_geometry.svg)

**Learning goals:** read `(B,C,T,H,W)` shapes, compute `Conv3d` output sizes, convert a token grid to a sequence, map flat indices to coordinates, and use gather/scatter safely. This notebook uses only synthetic data and runs on CPU.

In [ ]:
import random
import numpy as np
import torch

SEED = 7
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.set_num_threads(1)
device = torch.device('cpu')
print(f'torch={torch.__version__}, device={device}')

## 1. Name every axis

A video batch has shape `(batch, channel, time, height, width)`. An integer index removes an axis, while a slice keeps it. `permute` reorders axes; `reshape` only regroups the underlying sequence of values.

In [ ]:
B, C, T, H, W = 2, 3, 8, 16, 16
video = torch.arange(B*C*T*H*W, dtype=torch.float32).reshape(B, C, T, H, W)
frame = video[0, :, 2]                 # integer indices remove B and T
one_frame_batch = video[0:1, :, 2:3]   # slices retain singleton axes
assert frame.shape == (C, H, W)
assert one_frame_batch.shape == (1, C, 1, H, W)
channels_last = video.permute(0, 2, 3, 4, 1)
assert channels_last.shape == (B, T, H, W, C)
print('video:', tuple(video.shape), 'frame:', tuple(frame.shape))

## 2. Compute a tubelet grid

For input length $L$, kernel $K$, stride $S$, padding $P$, and dilation $A$, the output is $\lfloor(L+2P-A(K-1)-1)/S+1\rfloor$. Equal kernel and stride create non-overlapping tubelets. A learned filter maps each local block to one output feature.

In [ ]:
def conv_output(length, kernel, stride, padding=0, dilation=1):
    return (length + 2*padding - dilation*(kernel - 1) - 1) // stride + 1

D = 12
kernel = stride = (2, 4, 4)
embed = torch.nn.Conv3d(C, D, kernel_size=kernel, stride=stride, bias=False)
x = torch.randn(B, C, T, H, W, device=device)
grid = embed(x)
expected_grid = (B, D, conv_output(T, 2, 2), conv_output(H, 4, 4), conv_output(W, 4, 4))
assert tuple(grid.shape) == expected_grid == (2, 12, 4, 4, 4)
print('tubelet grid:', tuple(grid.shape))

## 3. Grid to sequence and back

`flatten(2)` merges the three grid axes but keeps batch and feature axes. `transpose(1,2)` puts features last for the `(B,N,D)` convention. These layout operations usually avoid arithmetic and often return views.

In [ ]:
Bt, Dt, Tt, Ht, Wt = grid.shape
tokens = grid.flatten(2).transpose(1, 2)
N = Tt * Ht * Wt
assert tokens.shape == (Bt, N, Dt)
restored = tokens.transpose(1, 2).reshape(Bt, Dt, Tt, Ht, Wt)
torch.testing.assert_close(restored, grid)
print('sequence:', tuple(tokens.shape), 'round trip exact:', torch.equal(restored, grid))

## 4. Flat token indices preserve coordinates

Width changes fastest. The flat index is `n = (t*H_grid + h)*W_grid + w`. Integer division and remainders invert the mapping.

In [ ]:
def flatten_coord(t, h, w, grid_h, grid_w):
    return (t * grid_h + h) * grid_w + w

def unflatten_index(n, grid_h, grid_w):
    t, remainder = divmod(n, grid_h * grid_w)
    h, w = divmod(remainder, grid_w)
    return t, h, w

for n in range(N):
    coord = unflatten_index(n, Ht, Wt)
    assert flatten_coord(*coord, Ht, Wt) == n
print('token 27 ->', unflatten_index(27, Ht, Wt))

## 5. Gather and scatter complete feature vectors

`torch.gather` requires an index tensor with the same rank as the input. `unsqueeze` adds a feature axis, and `expand` broadcasts it without allocating repeated index storage. `scatter_` restores values when indices are unique. For repeated indices, use `scatter_add_` or an explicit reduction.

In [ ]:
indices = torch.tensor([[0, 5, 27], [2, 11, 63]])
expanded = indices.unsqueeze(-1).expand(-1, -1, Dt)
selected = torch.gather(tokens, dim=1, index=expanded)
assert selected.shape == (B, 3, D)
canvas = torch.zeros_like(tokens)
canvas.scatter_(dim=1, index=expanded, src=selected)
for b in range(B):
    torch.testing.assert_close(canvas[b, indices[b]], tokens[b, indices[b]])
assert torch.count_nonzero(canvas).item() <= B * 3 * D
print('selected:', tuple(selected.shape), 'nonzero canvas entries:', torch.count_nonzero(canvas).item())

## Efficiency, exercises, and takeaways

`Conv3d` is far faster than Python loops over tubelets. Prefer `expand` to `repeat`, and call `contiguous()` only when an API requires contiguous storage. Assert shapes at every layout boundary.

**Exercises:** (1) Change the input to `(2,3,10,20,20)` and kernel/stride to `(2,5,5)`; predict the grid before running it. (2) Select five indices per sample and verify a gather/scatter round trip. (3) Create repeated indices and compare overwrite behavior with `scatter_add_`.

**Takeaways:** shape is meaning; tubelets create a learned local token grid; flattening preserves a known coordinate convention; gather selects and scatter places batched tokens efficiently.

## Continue learning

[Lecture](../lectures/01_spatiotemporal_tensor_geometry.md) | [Curriculum](../README.md) | [Next notebook: 02](02_inner_product_geometry.ipynb)